# 01: Fluid Dynamics & LFA Design Optimization

## Overview
This notebook implements a 1D capillary flow model based on an extension of Washburn's Law and Darcy's Law (Byers et al., 2020; Gosselin et al.), applied to the design of a three-channel, nanocatalyst-amplified Lateral Flow Immunoassay (LFIA). The goal is to determine the channel lengths (L1, L2, L3) required for each reagent to reach the test line at precisely timed intervals, enabling fully automated sequential reagent delivery.

## Device Design
The device comprises **three independent parallel channels**, each connected to the test line. L1, L2, L3 are the lengths from the far end of each respective channel to the test line:

| Channel | Reagent | Role |
|---------|---------|------|
| Sample channel | AuNP (proxy for PtNP) | Target binding |
| Buffer channel | PBS | Washing |
| Amplification channel | DAB | Signal amplification |

## Physical Model
According to [Byers et al.](https://doi.org/10.1021/acsomega.0c00115), capillary flow velocity $V$ in a rectangular nitrocellulose channel is:

$$V = k \cdot \frac{2\cos(\theta)(w+h)}{6LC} \quad \cdots (3)$$

where:
- $k = \gamma/\mu$ = ratio of surface tension to viscosity (reagent- and width-dependent)
- $\theta$ = advancing contact angle of pure water on nitrocellulose (40°)
- $w$, $h$ = channel width and membrane thickness
- $L$ = travel distance
- $C = 2h/w + w/h$ = geometric constant

Since $V = dL/dt$, the cumulative travel time over discrete distance increments is:

$$t = \sum \frac{\Delta L}{V(L)} \quad \cdots (4)$$

For channel widths (3 mm, 4 mm) not tabulated in Byers et al., $k$ is estimated by **linear interpolation** between the 1 mm and 5 mm values (Table S2, thesis):

$$\frac{k - k_1}{w - w_1} = \frac{k_i - k_1}{w_i - w_1} \quad \cdots (5)$$

## Assay Timeline (Table S1, thesis)
| Event | Reagent | Time target | Channel | Length |
|-------|---------|------------|---------|--------|
| Complexation reaches test line | AuNP | t = 120 s | Sample | **L1** |
| Buffer solution reaches test line | PBS (≈ DAB) | t = 300 s | Buffer | **L2** |
| Amplification solution reaches test line | DAB | t = 600 s | Amplification | **L3** |

**Note:** PBS is assumed to have the same surface tension and viscosity as DAB solution, as it contains neither surfactants nor nanoparticles and behaves as pure water.

**All $k$ values are from Table S3 of [Byers et al., 2020](https://doi.org/10.1021/acsomega.0c00115), with 3 mm and 4 mm values estimated by linear interpolation (Table S2, thesis).**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## Configuration
All physical constants and simulation parameters are defined here.

In [ ]:
# =============================================================================
# PHYSICAL CONSTANTS (from Byers et al., 2020)
# =============================================================================
THETA_DEG  = 40     # Advancing contact angle of pure water on 0.45 µm nitrocellulose (degrees)
H_MM       = 0.1    # Membrane thickness (mm) — FF80HP / HF135, 0.45 µm pore size
theta      = np.radians(THETA_DEG)

# Distance array: 1 to 110 mm in 1 mm steps

DIST_STEP  = 1
dists      = np.arange(1, 111, DIST_STEP, dtype=float)

# =============================================================================
# k VALUES (surface tension / viscosity ratio) — Byers et al. Table S3
# Reagent: AuNP solution (proxy for PtNP; nanoparticle dominates surface tension/viscosity)
# =============================================================================
K_AuNP_KNOWN = {5: 230.3, 2: 126.1, 1: 75.16}

# Reagent: DAB solution (also used as proxy for PBS — assumed same as pure water)
K_DAB_KNOWN  = {5: 326.4, 2: 137.2, 1: 52.76}

# =============================================================================
# ASSAY TIMELINE — Table S1, thesis
# =============================================================================
T_L1 = 120   # AuNP reaches test line via sample channel
T_L2 = 300   # PBS reaches test line via buffer channel
T_L3 = 600   # DAB reaches test line via amplification channel

print("Configuration loaded.")
print(f"  Contact angle: {THETA_DEG}° | Membrane thickness: {H_MM} mm")
print(f"  Distance range: {dists[0]:.0f}–{dists[-1]:.0f} mm")
print(f"  Time targets: L1={T_L1}s | L2={T_L2}s | L3={T_L3}s")

## Helper Functions

In [ ]:
def washburn_velocity(k, w, z_array):
    """Compute capillary flow velocity at each distance L (Eq. 3, thesis).

    Parameters
    ----------
    k       : float   — surface tension/viscosity ratio for the reagent and channel width
    w       : float   — channel width (mm)
    z_array : ndarray — distance array (mm)

    Returns
    -------
    vel : ndarray — velocity at each distance (mm/s)
    """
    C = 2 * H_MM / w + w / H_MM
    return k * 2 * np.cos(theta) * (w + H_MM) / (6 * z_array * C)


def cumulative_time(k, w):
    """Compute cumulative travel time at each distance step (Eq. 4, thesis).

    Uses discrete summation: t = cumsum(ΔL / V(L))
    """
    vel = washburn_velocity(k, w, dists)
    return np.cumsum(DIST_STEP / vel)


def interpolate_k(w_target, k_known):
    """Estimate k for a non-tabulated channel width by linear interpolation (Eq. 5, thesis).

    Parameters
    ----------
    w_target : float — target channel width (mm)
    k_known  : dict  — {width: k_value} for tabulated widths from Byers et al.
    """
    widths = sorted(k_known.keys())
    k_vals = [k_known[w] for w in widths]
    return float(np.interp(w_target, widths, k_vals))


def find_channel_length(t_target, k, w):
    """Return the distance (mm) a reagent travels in t_target seconds."""
    tol_time = cumulative_time(k, w)
    return float(np.interp(t_target, tol_time, dists))


print("Helper functions defined.")

---
## Section 1 — Model Validation: Reproducing Byers et al. Results
Simulating V vs L and L vs t for AuNP and DAB at known channel widths (1, 2, 5 mm) to validate the model.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Model Validation: Washburn Flow Simulation (Byers et al., 2020)", fontsize=13)

reagents = {"AuNP": K_AuNP_KNOWN, "DAB": K_DAB_KNOWN}

for col, (reagent, k_dict) in enumerate(reagents.items()):
    for w in sorted(k_dict.keys(), reverse=True):
        k     = k_dict[w]
        vel   = washburn_velocity(k, w, dists)
        tol_t = cumulative_time(k, w)

        axes[0, col].plot(dists, vel, label=f"{w} mm")
        axes[1, col].plot(tol_t, dists, label=f"{w} mm")

    axes[0, col].set_title(f"Velocity vs Distance ({reagent})")
    axes[0, col].set_xlabel("Distance (mm)")
    axes[0, col].set_ylabel("Velocity (mm/s)")
    axes[0, col].set_xlim(0, 100)
    axes[0, col].legend()
    axes[0, col].grid(alpha=0.3)

    axes[1, col].set_title(f"Distance vs Time ({reagent})")
    axes[1, col].set_xlabel("Time (s)")
    axes[1, col].set_ylabel("Distance (mm)")
    axes[1, col].legend()
    axes[1, col].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Section 2 — Linear Interpolation for 3 mm and 4 mm Channels
Estimating $k$ values for channel widths not reported in Byers et al. using Eq. 5 (thesis, Table S2).

In [ ]:
# Build full k tables for 1–5 mm (Table S2, thesis)
K_AuNP_ALL = {w: interpolate_k(w, K_AuNP_KNOWN) for w in [5, 4, 3, 2, 1]}
K_DAB_ALL  = {w: interpolate_k(w, K_DAB_KNOWN)  for w in [5, 4, 3, 2, 1]}

print("Table S2 (thesis) — k values for AuNP solution:")
for w in sorted(K_AuNP_ALL.keys(), reverse=True):
    src = "Byers et al." if w in K_AuNP_KNOWN else "interpolated"
    print(f"  k{w} = {K_AuNP_ALL[w]:.1f}   ({src})")

print("\nTable S2 (thesis) — k values for DAB / PBS solution:")
for w in sorted(K_DAB_ALL.keys(), reverse=True):
    src = "Byers et al." if w in K_DAB_KNOWN else "interpolated"
    print(f"  k{w} = {K_DAB_ALL[w]:.1f}   ({src})")

In [ ]:
# Visualise flow curves for all 5 widths — reproducing Figures 9 & 10 of thesis
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Washburn Flow Simulation — All Channel Widths (Figures 9 & 10, thesis)", fontsize=12)

for w in sorted(K_AuNP_ALL.keys(), reverse=True):
    vel   = washburn_velocity(K_AuNP_ALL[w], w, dists)
    tol_t = cumulative_time(K_AuNP_ALL[w], w)
    ls = "--" if w in [3, 4] else "-"
    lbl = f"{w} mm" + ("*" if w in [3, 4] else "")
    axes[0, 0].plot(dists, vel, linestyle=ls, label=lbl)
    axes[1, 0].plot(tol_t, dists, linestyle=ls, label=lbl)

for w in sorted(K_DAB_ALL.keys(), reverse=True):
    vel   = washburn_velocity(K_DAB_ALL[w], w, dists)
    tol_t = cumulative_time(K_DAB_ALL[w], w)
    ls = "--" if w in [3, 4] else "-"
    lbl = f"{w} mm" + ("*" if w in [3, 4] else "")
    axes[0, 1].plot(dists, vel, linestyle=ls, label=lbl)
    axes[1, 1].plot(tol_t, dists, linestyle=ls, label=lbl)

titles = ["Velocity vs Distance (AuNP)", "Velocity vs Distance (PBS & DAB)",
          "Distance vs Time (AuNP)",      "Distance vs Time (PBS & DAB)"]
for i, ax in enumerate(axes.flat):
    ax.set_title(titles[i] + ("  (* = interpolated)" if i in [0,1] else ""))
    ax.set_xlabel("Distance (mm)" if i < 2 else "Time (s)")
    ax.set_ylabel("Velocity (mm/s)" if i < 2 else "Distance (mm)")
    if i < 2:
        ax.set_xlim(0, 100)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Section 3 — Determining Channel Lengths L1, L2, L3
Using `np.interp` to read off the travel distance at each assay time target.

In [ ]:
widths = [5, 4, 3, 2, 1]

results = {}
for w in widths:
    results[w] = {
        "L1": find_channel_length(T_L1, K_AuNP_ALL[w], w),  # AuNP, 120 s, sample channel
        "L2": find_channel_length(T_L2, K_DAB_ALL[w],  w),  # PBS,  300 s, buffer channel
        "L3": find_channel_length(T_L3, K_DAB_ALL[w],  w),  # DAB,  600 s, amplification channel
    }

print("Simulation results (reproducing Table 1, thesis):")
print(f"\n{'Width':>7}  {'L1 (mm)':>10}  {'L2 (mm)':>10}  {'L3 (mm)':>10}")
print("-" * 45)
for w in widths:
    r   = results[w]
    tag = " *" if w in [3, 4] else "  "
    print(f"{w:>5}mm{tag}  {r['L1']:>10.2f}  {r['L2']:>10.2f}  {r['L3']:>10.2f}")

print("\n* k estimated by linear interpolation")
print("\nThesis Table 1 reference values:")
thesis = {5:[37.43,70.89,100.46], 4:[34.16,63.11,89.46], 3:[30.56,52.23,76.90], 2:[27.92,46.36,65.77], 1:[21.79,29.03,41.26]}
for w in widths:
    t = thesis[w]
    print(f"  {w}mm: L1={t[0]}, L2={t[1]}, L3={t[2]}")

---
## Section 4 — Design Conclusion (Thesis Figure 11)
From the simulation results (Table 1, thesis), a **4 mm wide device** was selected because it produces a relatively fast and consistent flow compared to 5 mm while yielding channel lengths that are practical for manufacturing (comparable to commercial lateral flow strip dimensions).

The final device dimensions for the 4 mm channel design are:

| Channel | Reagent | Time | Length |
|---------|---------|------|--------|
| Sample (L1) | AuNP | 120 s | **35 mm** |
| Buffer (L2) | PBS | 300 s | **64 mm** |
| Amplification (L3) | DAB | 600 s | **90 mm** |

(Rounded to nearest mm for fabrication; computed values: L1=34.16, L2=63.11, L3=89.46 mm)

In [ ]:
# Reproduce Table 1 as a bar chart
labels  = ["L1\n(AuNP, 120 s)\nSample channel",
           "L2\n(PBS, 300 s)\nBuffer channel",
           "L3\n(DAB, 600 s)\nAmplification channel"]
keys    = ["L1", "L2", "L3"]
colours = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]

fig, ax = plt.subplots(figsize=(10, 5))
x     = np.arange(len(keys))
bar_w = 0.15

for i, w in enumerate(widths):
    vals  = [results[w][k] for k in keys]
    hatch = "//" if w in [3, 4] else ""
    label = f"{w} mm" + ("*" if w in [3, 4] else "")
    ax.bar(x + i * bar_w, vals, bar_w, label=label, color=colours[i], hatch=hatch, edgecolor="white")

# Highlight the selected 4 mm design
for j, key in enumerate(keys):
    val = results[4][key]
    ax.annotate(f"{val:.1f}", xy=(x[j] + 1 * bar_w, val), xytext=(0, 4),
                textcoords="offset points", ha="center", fontsize=8, color="#55A868",
                fontweight="bold")

ax.set_xticks(x + bar_w * 2)
ax.set_xticklabels(labels)
ax.set_ylabel("Channel Length (mm)")
ax.set_title("Simulated Channel Lengths L1–L3 for All Channel Widths (Table 1, thesis)")
ax.legend(title="Channel Width", loc="upper left")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("Selected design: 4 mm wide channel (highlighted in green above)")
print(f"  L1 = {results[4]['L1']:.2f} mm  →  fabricated as 35 mm")
print(f"  L2 = {results[4]['L2']:.2f} mm  →  fabricated as 64 mm")
print(f"  L3 = {results[4]['L3']:.2f} mm  →  fabricated as 90 mm")
print(f"  w  = 4 mm")